# SpikeDistill-RUL — data preparation

Executed first implementation milestone. Read README.md for the complete 12-step roadmap. Models and research results are not implemented yet. Original files remain in ../CMaps.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from prepare_data import load_trajectories, prepare_subset
PROJECT = Path.cwd()
DATA = PROJECT.parent / "CMaps"
ARTIFACTS = PROJECT / "artifacts" / "data"
assert DATA.is_dir(), "Open this notebook from the SpikeDistill-RUL directory"
print("Dataset:", DATA)
print("Artifacts:", ARTIFACTS)

## 1. Raw dataset audit
An engine is a separate time series. Never mix windows from the same engine across training and validation. The numeric files, rather than a hard-coded README count, determine actual counts.

In [ ]:
summary = json.loads((ARTIFACTS / "audit_summary.json").read_text())
rows = []
for s in summary:
    rows.append({"subset": s["subset"], "raw_train_engines": s["official_train"]["engines"], "raw_test_engines": s["official_test"]["engines"], "training_engines": s["training_engines"], "validation_engines": s["validation_engines"], "features": s["features"], "training_windows": s["arrays"]["train"]["shape"][0]})
print(pd.DataFrame(rows).to_string(index=False))
print("FD004 discrepancy: bundled README says 249 test engines; supplied files have 248 aligned engines/labels.")

## 2. RUL labels and reproducible split
For training histories, raw RUL = last failure cycle - current cycle. y = min(raw RUL, 125). Test histories stop before failure; their labels come from RUL_FD001.txt. Test labels never enter preprocessing fitting. Validation uses fixed incomplete prefixes from held-out training engines.

In [ ]:
manifest = json.loads((ARTIFACTS / "FD001" / "split_manifest.json").read_text())
assert not (set(manifest["train_engine_ids"]) & set(manifest["validation_engine_ids"]))
print("Seed:", manifest["seed"], "Window:", manifest["window"], "Cap:", manifest["rul_cap"])
print("Training engines:", len(manifest["train_engine_ids"]))
print("Validation engines:", len(manifest["validation_engine_ids"]))
print("Validation protocol:", manifest["validation_protocol"])

## 3. Train-only normalization
Normalization means subtracting a fitted mean and dividing by a fitted standard deviation. Validation/test never determine those values. Multi-condition subsets use separate sensor statistics per learned operating regime.

In [ ]:
prep = json.loads((ARTIFACTS / "FD001" / "preprocessing.json").read_text())
print("FD001 input features:", prep["feature_names"])
print("Dropped near-constant sensors:", prep["dropped_sensors"])
print("Normalization:", prep["normalization"])

## 4. Sliding windows and masks
A training input is 30 observed cycles by the feature count. One RUL label belongs to the last cycle of that window. Padding masks distinguish synthetic repeated rows from observed history. Retain masks in downstream models.

In [ ]:
with np.load(ARTIFACTS / "FD001" / "train.npz", allow_pickle=False) as data:
    print("X:", data["X"].shape, data["X"].dtype)
    print("First example engine / endpoint:", data["engine_id"][0], data["end_cycle"][0])
    print("First example capped / raw label:", data["y"][0], data["y_raw"][0])
    print("All features finite:", np.isfinite(data["X"]).all())
    assert data["X"].shape[:2] == data["mask"].shape
for subset in ["FD001", "FD002", "FD003", "FD004"]:
    with np.load(ARTIFACTS / subset / "test.npz", allow_pickle=False) as data:
        print(subset, "test windows:", len(data["X"]), "padded windows:", int((~data["mask"].all(axis=1)).sum()))

## Rebuild prepared data when needed
The preparation command is in README.md. Existing artifacts are reused above to keep notebook inspection fast. To regenerate inside Python, run the following manually:

```python
for subset in ["FD001", "FD002", "FD003", "FD004"]:
    prepare_subset(DATA, ARTIFACTS, subset, window=30, cap=125, seed=42)
```

Next: exploratory plots, the training environment, and conventional baselines. No model-quality or hardware-efficiency claim can be made from these data checks.